In [1]:
import os
from pathlib import Path
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
from perf_estimator.trainer.plugins import ProfilerCallback, SnapshotCallback
from ures.string import format_memory

/home/glaswigian/miniconda3/envs/Huggingface/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [3]:
# Configure logging
# ---------------------------
# 1. Model Initialization (from scratch)
# ---------------------------
# We use the configuration of a popular small-scale LLM (facebook/opt-125m)
# but initialize the model randomly (i.e. train from scratch)
model_name = "facebook/opt-125m"
model_name = "EleutherAI/gpt-neo-125M"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
config = AutoConfig.from_pretrained(model_name)  # load config; do NOT load pretrained weights
model = AutoModelForCausalLM.from_config(config)   # randomly initialized model
print(model)

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

In [4]:
model_size = 0
parameters_list = list(model.parameters())
parameters_list.reverse()
for tensor in parameters_list:
    para_size = tensor.nelement() * tensor.element_size()
    model_size += para_size
    print(f"{tensor.shape}: size: {format_memory(para_size)}")
print(f"Model size: {format_memory(model_size)}")

torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 3072]): size: 9.00 MB
torch.Size([3072]): size: 12.00 KB
torch.Size([3072, 768]): size: 9.00 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 3072]): size: 9.00 MB
torch.Size([3072]): size: 12.00 KB
torch.Size([3072, 768]): size: 9.00 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Si

In [5]:
# Load tokenizer (we can reuse the pretrained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # assign PAD token if missing

# Enable gradient checkpointing to reduce memory usage (at the cost of additional compute)
model.gradient_checkpointing_enable()

# ---------------------------
# 2. Dataset Preparation
# ---------------------------
# Load the Wikitext-2 dataset as our general-purpose text corpus.
# For a quick profiling run, we use only a small subset.
dataset = load_dataset(
    "wikitext", "wikitext-2-raw-v1",
)
train_dataset = dataset["train"].select(range(1000))  # limit to 1000 examples for this demo

# Tokenization: convert text to token IDs (truncated to a maximum length)
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator: handles padding and prepares labels for causal LM (labels equal to input_ids)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ---------------------------
# 3. Trainer Setup with Memory Profiling Callback
# ---------------------------
# TrainingArguments are set to run only 3 steps and use a small batch size suitable for an 8–12GB GPU.
training_args = TrainingArguments(
    output_dir="output",
    # per_device_train_batch_size=2,
    max_steps=3,  # run only 3 training iterations for profiling
    gradient_accumulation_steps=1,
    fp16=False,  # using full precision; set True if your GPU supports mixed precision to save memory
    logging_steps=1,
    report_to=[],  # disable external logging (e.g., wandb)
    disable_tqdm=False,
    use_cpu=False,
    do_train=True,
    do_eval=False,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[ProfilerCallback(), SnapshotCallback()],
)

# ---------------------------
# 4. Training with PyTorch Profiler
# ---------------------------

trainer.train()  # run 3 training iterations

# Print a summary of the profiler's memory usage by CUDA operation (top 10 ops)
print("Profiler Memory Usage Summary (top CUDA ops):")


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Starting profiler...


Step,Training Loss
1,10.633300
2,10.422700
3,9.902600


Stopping profiler...
Profiler Memory Usage Summary (top CUDA ops):


In [14]:
from ures.files import filter_files
p_file = filter_files("pt.trace.json", str(Path().home().joinpath("DL-Estimator")), fuzz=True)[1]
p_file


'/home/glaswigian/DL-Estimator/EleutherAI-gpt-neo-125M-cpu/results/callback/Profiler/Glaswigian-Researcher_47245.1741794696760298726.pt.trace.json'

In [15]:
from perf_estimator.estimator import TrainerEstimator
from perf_estimator.dataset import image_dataset
from perf_estimator.config import Config
_config = Config()
_config.trainer.huggingface_enable = True
_config.trainer.huggingface_model_name = model_name
estimator = TrainerEstimator(
    dataloader=image_dataset(batch=100),
    profiler_file=p_file,
    max_gpu_memory_in_gb=8,
    config=_config,
)
allocator, est_result = estimator.estimate()

Duplicate layer name found: GPTNeoBlock_11. Renaming to GPTNeoBlock_11_30
Duplicate layer name found: LayerNorm_22. Renaming to LayerNorm_22_f5
Duplicate layer name found: GPTNeoAttention_11. Renaming to GPTNeoAttention_11_66
Duplicate layer name found: GPTNeoSelfAttention_11. Renaming to GPTNeoSelfAttention_11_67
Duplicate layer name found: Linear_66. Renaming to Linear_66_87
Duplicate layer name found: Linear_67. Renaming to Linear_67_73
Duplicate layer name found: Linear_68. Renaming to Linear_68_98
Duplicate layer name found: Dropout_34. Renaming to Dropout_34_98
Duplicate layer name found: Linear_69. Renaming to Linear_69_db
Duplicate layer name found: Dropout_35. Renaming to Dropout_35_84
Duplicate layer name found: LayerNorm_23. Renaming to LayerNorm_23_22
Duplicate layer name found: GPTNeoMLP_11. Renaming to GPTNeoMLP_11_50
Duplicate layer name found: Linear_70. Renaming to Linear_70_2c
Duplicate layer name found: NewGELUActivation_11. Renaming to NewGELUActivation_11_fa
Duplic

In [16]:
allocator.plot_memory_change()